# Feature Engineering for Logistic Regression Model – Phishing Email Detection

This notebook performs feature engineering for a Logistic Regression (LR) model to classify phishing emails. It uses the publicly available **Kaggle Phishing Email Dataset**, as recommended, due to its better structure and labeling quality compared to the internal UVic dataset.

The notebook focuses on:
- Proper data splitting (training, validation, test) to avoid data leakage.
- Extraction of both structured features and TF-IDF representations from the training set only.
- Preprocessing and transformation pipelines that ensure separation between training and evaluation phases.

Phishing emails often show linguistic, structural, and behavioral indicators like:
- Excessive capitalization in subject lines
- Suspicious URLs or attachments
- Emotional or urgent language
- Presence of special characters and exclamations

Based on insights from the literature, we will:
- Extract structured features from the email content
- Apply TF-IDF and Bag-of-Words vectorization techniques
- Save the prepared datasets for model training and evaluation

This feature engineering step is designed to support a Logistic Regression baseline model and allow fair comparison with alternative vectorization strategies and later embedding-based models.

## 1. Load Cleaned Dataset
- Source Kaggle Dataset: phishing_email.csv

In [2]:
# Import required libraries
import pandas as pd        # Data handling and manipulation
import numpy as np         # Numerical operations and array structures
import matplotlib.pyplot as plt  # Plotting basic charts (e.g., histograms, bar plots)
import seaborn as sns      # Advanced statistical visualizations

# Load the Kaggle phishing email dataset
df = pd.read_csv("phishing_email.csv", encoding="utf-8-sig")

# Display basic dataset information
print(f"Dataset shape: {df.shape}")    # Number of rows and columns
print("Column names:", list(df.columns))  # List of column names

# Preview the first few rows
df.head()

Dataset shape: (82486, 2)
Column names: ['text_combined', 'label']


,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


### 2. Basic Structural Features

Since the dataset provides a single combined text field (`text_combined`), we will extract simple structural features from this full text:

- `text_length`: Total number of characters in the email.
- `num_exclamations`: Number of exclamation marks, often used to create urgency.
- `num_links`: Count of URL-like patterns (e.g., "http", "www").
- `num_uppercase_words`: Number of fully capitalized words.

In [3]:
# --- Feature: text_length ---
# Total number of characters in the full email text
df['text_length'] = df['text_combined'].apply(lambda x: len(str(x)))

# --- Feature: num_exclamations ---
# Number of exclamation marks in the email (e.g., "!!!" indicates urgency)
df['num_exclamations'] = df['text_combined'].apply(lambda x: str(x).count('!'))

# --- Feature: num_links ---
# Approximate count of links by checking common URL patterns (e.g., "http", "www")
df['num_links'] = df['text_combined'].apply(lambda x: x.lower().count('http') + x.lower().count('www'))

# --- Feature: num_uppercase_words ---
# Count the number of words in all uppercase (potentially shouting or emphasis)
df['num_uppercase_words'] = df['text_combined'].apply(
    lambda x: sum(1 for word in str(x).split() if word.isupper())
)

# Preview the new features
df[['text_length', 'num_exclamations', 'num_links', 'num_uppercase_words']].head()

,text_length,num_exclamations,num_links,num_uppercase_words
0,65,0,0,0
1,1071,0,0,0
2,148,0,0,0
3,65,0,0,0
4,65,0,0,0


### 3. Train/Validation/Test Split

We split the dataset into training (80%), validation (10%), and test (10%) sets using stratified sampling to preserve class distribution. This ensures that the feature extraction step (e.g., TF-IDF or BOW) is applied only on the training set, avoiding data leakage.

In [5]:
from sklearn.model_selection import train_test_split

# Extract features and labels
X_raw = df['text_combined']
y = df['label']

# First split: 80% train, 20% temp (val + test)
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=42
)

# Second split: 50% val, 50% test from remaining 20%
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

# Confirm distribution
print("Train set:", y_train.value_counts(normalize=True))
print("Val set:  ", y_val.value_counts(normalize=True))
print("Test set: ", y_test.value_counts(normalize=True))

Train set: label
1    0.519973
0    0.480027
Name: proportion, dtype: float64
Val set:   label
1    0.520063
0    0.479937
Name: proportion, dtype: float64
Test set:  label
1    0.519942
0    0.480058
Name: proportion, dtype: float64


### 4. TF-IDF Vectorization

We convert the raw email text into numerical vectors using TF-IDF. To avoid data leakage, we fit the vectorizer only on the training data and use it to transform the validation and test sets.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF vectorizer
# - max_features limits the vocabulary size
# - ngram_range=(1,2) captures unigrams and bigrams
tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

# Fit only on the training set
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_raw)

# Transform validation and test sets using the trained vectorizer
X_val_tfidf = tfidf_vectorizer.transform(X_val_raw)
X_test_tfidf = tfidf_vectorizer.transform(X_test_raw)

# Confirm the shapes of the vectorized data
print("TF-IDF matrix shapes:")
print("Train:", X_train_tfidf.shape)
print("Val:  ", X_val_tfidf.shape)
print("Test: ", X_test_tfidf.shape)

TF-IDF matrix shapes:
Train: (65988, 5000)
Val:   (8249, 5000)
Test:  (8249, 5000)


### 5. Save Processed Data and TF-IDF Vectorizer

We save the vectorized train/validation/test sets, their corresponding labels, and the trained TF-IDF vectorizer using `joblib`. This allows us to reuse the preprocessed data in the model training notebook without repeating the transformation steps.

In [10]:
import joblib

# Save TF-IDF vectorized feature matrices
joblib.dump(X_train_tfidf, 'X_train_tfidf.pkl')  # Sparse matrix for training features
joblib.dump(X_val_tfidf, 'X_val_tfidf.pkl')      # Sparse matrix for validation features
joblib.dump(X_test_tfidf, 'X_test_tfidf.pkl')    # Sparse matrix for test features

# Save labels for each split
joblib.dump(y_train, 'y_train.pkl')  # Corresponding labels for training set
joblib.dump(y_val, 'y_val.pkl')      # Corresponding labels for validation set
joblib.dump(y_test, 'y_test.pkl')    # Corresponding labels for test set

# Save the trained TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, 'tfidf_vectorizer.pkl')  # This ensures consistent transformation later

print("All data and the vectorizer have been saved successfully.")

All data and the vectorizer have been saved successfully.


### 6. Bag-of-Words (BOW) Vectorization

We now apply CountVectorizer (Bag-of-Words) as an alternative to TF-IDF. This will allow us to compare the performance of Logistic Regression when using different text representation methods. As with TF-IDF, we train the vectorizer only on the training set to prevent data leakage.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the BOW vectorizer
# - max_features limits the vocabulary to the most frequent words/ngrams
# - ngram_range=(1,2) captures unigrams and bigrams
bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1,2))

# Fit the vectorizer only on the training set
X_train_bow = bow_vectorizer.fit_transform(X_train_raw)

# Transform validation and test sets
X_val_bow = bow_vectorizer.transform(X_val_raw)
X_test_bow = bow_vectorizer.transform(X_test_raw)

# Print shape of the BOW matrices
print("Bag-of-Words matrix shapes:")
print("Train:", X_train_bow.shape)
print("Val:  ", X_val_bow.shape)
print("Test: ", X_test_bow.shape)

Bag-of-Words matrix shapes:
Train: (65988, 5000)
Val:   (8249, 5000)
Test:  (8249, 5000)


### 7. Save Bag-of-Words Feature Matrices and Vectorizer

We save the BOW-transformed datasets and the fitted vectorizer so they can be used for model training and evaluation in the Logistic Regression notebook.

In [9]:
import joblib

# Save BOW feature matrices
joblib.dump(X_train_bow, 'X_train_bow.pkl')
joblib.dump(X_val_bow, 'X_val_bow.pkl')
joblib.dump(X_test_bow, 'X_test_bow.pkl')

# Save the BOW vectorizer for future use or inspection
joblib.dump(bow_vectorizer, 'bow_vectorizer.pkl')

print(" Bag-of-Words BOW features and vectorizer saved successfully.")

 Bag-of-Words BOW features and vectorizer saved successfully.
